In [13]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import KBinsDiscretizer



Binning 

In [10]:
ages = pd.DataFrame({"name":["Krish", "Dhruv", "Vraj", "Heer", "Priya", "Hasti", "Jasi", "Riya"],"age": [5, 18, 22, 35, 45, 60, 72, 90]})
print(ages)



    name  age
0  Krish    5
1  Dhruv   18
2   Vraj   22
3   Heer   35
4  Priya   45
5  Hasti   60
6   Jasi   72
7   Riya   90


In [11]:
# Equal-width bins
bin_width = KBinsDiscretizer(n_bins=4, encode="ordinal", strategy="uniform")
ages["age_bin_width"] = bin_width.fit_transform(ages[["age"]])

# Equal-frequency (quantile) bins
bin_freq = KBinsDiscretizer(n_bins=4, encode="ordinal", strategy="quantile")
ages["age_bin_freq"] = bin_freq.fit_transform(ages[["age"]])

# Manual, human-readable bins
ages["age_group"] = pd.cut(
    ages["age"], bins=[0, 12, 19, 39, 59, 120],
    labels=["child", "teen", "adult", "middle_age", "senior"]
)

In [12]:
print(ages)

    name  age  age_bin_width  age_bin_freq   age_group
0  Krish    5            0.0           0.0       child
1  Dhruv   18            0.0           0.0        teen
2   Vraj   22            0.0           1.0       adult
3   Heer   35            1.0           1.0       adult
4  Priya   45            1.0           2.0  middle_age
5  Hasti   60            2.0           2.0      senior
6   Jasi   72            3.0           3.0      senior
7   Riya   90            3.0           3.0      senior


Date_time_featuring

In [14]:

df = pd.DataFrame({"ts": pd.to_datetime([
    "2024-01-15 08:30:00", "2024-06-01 23:10:00", "2024-12-31 12:00:00"
])})

df["year"] = df["ts"].dt.year
df["month"] = df["ts"].dt.month
df["day"] = df["ts"].dt.day
df["hour"] = df["ts"].dt.hour
df["dayofweek"] = df["ts"].dt.dayofweek        # 0=Monday
df["is_weekend"] = df["dayofweek"].isin([5, 6]).astype(int)

# Cyclical encoding for month (period = 12) and hour (period = 24)
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

# Elapsed time features
df["days_since_epoch"] = (df["ts"] - pd.Timestamp("1970-01-01")).dt.days

In [15]:
print(df)

                   ts  year  month  day  hour  dayofweek  is_weekend  \
0 2024-01-15 08:30:00  2024      1   15     8          0           0   
1 2024-06-01 23:10:00  2024      6    1    23          5           1   
2 2024-12-31 12:00:00  2024     12   31    12          1           0   

      month_sin  month_cos      hour_sin  hour_cos  days_since_epoch  
0  5.000000e-01   0.866025  8.660254e-01 -0.500000             19737  
1  1.224647e-16  -1.000000 -2.588190e-01  0.965926             19875  
2 -2.449294e-16   1.000000  1.224647e-16 -1.000000             20088  


Ratio

In [16]:
df = pd.DataFrame({
    "price": [200000, 350000, 500000],
    "sqft": [1000, 1500, 2000],
    "income": [40000, 60000, 90000],
    "debt": [10000, 30000, 20000],
})

# Manual, domain-driven ratios
df["price_per_sqft"] = df["price"] / df["sqft"]
df["debt_to_income"] = df["debt"] / df["income"]

print(df)

    price  sqft  income   debt  price_per_sqft  debt_to_income
0  200000  1000   40000  10000      200.000000        0.250000
1  350000  1500   60000  30000      233.333333        0.500000
2  500000  2000   90000  20000      250.000000        0.222222


One hot vs Odinal 

In [27]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, Normalizer
import pandas as pd

df = pd.DataFrame({"income": [30000, 45000, 60000, 1000000]})  # note the outlier

standard = StandardScaler().fit_transform(df)     # mean=0, std=1
minmax = MinMaxScaler().fit_transform(df)         # scaled to [0, 1]
robust = RobustScaler().fit_transform(df)         # uses median/IQR, resists outliers

In [28]:
print(df)

    income
0    30000
1    45000
2    60000
3  1000000


In [29]:
print(standard)

[[-0.61342184]
 [-0.57716045]
 [-0.54089906]
 [ 1.73148135]]


In [30]:
print(minmax)

[[0.        ]
 [0.01546392]
 [0.03092784]
 [1.        ]]


In [31]:
print(robust)

[[-0.08866995]
 [-0.02955665]
 [ 0.02955665]
 [ 3.73399015]]


In [32]:
import pandas as pd

df = pd.DataFrame({
    "Name": ["Krish", "Dhruv", "Vraj", "Heer"],
    "City": ["Mumbai", "Delhi", "Surat", "Mumbai"]
})

print(df)

    Name    City
0  Krish  Mumbai
1  Dhruv   Delhi
2   Vraj   Surat
3   Heer  Mumbai


One hot encoding 

In [33]:
encoded_df = pd.get_dummies(df, columns=["City"])

print(encoded_df)

    Name  City_Delhi  City_Mumbai  City_Surat
0  Krish       False         True       False
1  Dhruv        True        False       False
2   Vraj       False        False        True
3   Heer       False         True       False


For integer (0,1)

In [35]:
encoded_df = pd.get_dummies(df, columns=["City"], dtype=int)
print(encoded_df)

    Name  City_Delhi  City_Mumbai  City_Surat
0  Krish           0            1           0
1  Dhruv           1            0           0
2   Vraj           0            0           1
3   Heer           0            1           0


In [36]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False)

encoded = encoder.fit_transform(df[["City"]])

encoded_df = pd.DataFrame(
    encoded,
    columns=encoder.get_feature_names_out(["City"])
)

print(encoded_df)

   City_Delhi  City_Mumbai  City_Surat
0         0.0          1.0         0.0
1         1.0          0.0         0.0
2         0.0          0.0         1.0
3         0.0          1.0         0.0


Odinal

In [37]:
import pandas as pd

df = pd.DataFrame({
    "Student": ["A", "B", "C", "D"],
    "Performance": ["Poor", "Average", "Good", "Excellent"]
})

print(df)

  Student Performance
0       A        Poor
1       B     Average
2       C        Good
3       D   Excellent


In [38]:
shirt = pd.DataFrame({
    "Size": ["Small", "Medium", "Large", "Medium", "Small"]
})

encoder = OrdinalEncoder(
    categories=[["Small", "Medium", "Large"]]
)

shirt["Size"] = encoder.fit_transform(shirt[["Size"]])

print(shirt)

   Size
0   0.0
1   1.0
2   2.0
3   1.0
4   0.0
